# **基于静态 Tensor 的 CV 融合算子开发**

## 概述

本小节介绍如何使用 Ascend C 基础API 的**静态 Tensor 编程模式**开发一个 Cube-Vector（CV）融合算子。课程围绕 `MMAD + LeakyRelu` 主线展开。

### 学习前置要求

学习本小节前，建议已经具备以下基础：

- 当前环境已安装 CANN Toolkit；
- 了解矩阵乘法的基本形状关系，即 `[M, K] × [K, N] → [M, N]`；
- 了解基本的AscendC算子开发和执行流程，包括device内存分配，H2D等操作；
- 已经学习静态Tensor基础课程，掌握基本的静态Tensor编程方式，会正确插入核内同步；

### 学习目标

完成本小节后，开发者应能够：

1. 熟练掌握**静态 Tensor 编程**方式的Buffer管理方法；
2. 深入理解昇腾NPU中硬件模块的同步机制；
3. 了解昇腾NPU分离架构中cube核和vector核的协作方式，学会写出核间同步算子；

### 本节内容

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">章节</th>
      <th align="left">内容</th>
      <th align="left">学习期望</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>环境准备</td>
      <td>初始化 CANN 环境</td>
      <td>初始化环境正常</td>
    </tr>
    <tr>
      <td>算子计算分析</td>
      <td>分析 <code>MMAD + LeakyRelu</code> 输入输出、核数比例和数据形状</td>
      <td>了解算子背景</td>
    </tr>
    <tr>
      <td>流水同步规则</td>
      <td>说明 AIC/AIV 搬运计算、流水和同步事件</td>
      <td>深入理解同步规则</td>
    </tr>
    <tr>
      <td>静态 Tensor MMAD</td>
      <td>阅读并运行纯 Cube MMAD 样例</td>
      <td>能理解 Cube 侧静态 Tensor 编程</td>
    </tr>
    <tr>
      <td>静态 Tensor LeakyRelu</td>
      <td>阅读并运行纯 Vector LeakyRelu 样例</td>
      <td>能理解 Vector 侧静态 Tensor 编程</td>
    </tr>
    <tr>
      <td>CV 融合实现</td>
      <td>阅读并运行 <code>MMAD + LeakyRelu</code> 融合样例</td>
      <td>能理解 AIC/AIV 协同计算编程</td>
    </tr>
    <tr>
      <td>课后实践</td>
      <td>尝试编写 Residual 输入并融合 <code>Add + LeakyRelu</code>的算子</td>
      <td>能独立完成更贴近网络结构的多输入 CV 融合变体</td>
    </tr>
  </tbody>
</table>

### 运行环境与硬件说明

- 本节运行脚本支持 `--mode=npu` 和 `--mode=sim`。有 NPU 环境时使用 `npu` 模式；无 NPU 环境时使用 `sim` 模式。
- 本节运行脚本支持 `--npu-arch=2201` 和 `--npu-arch=3510`，根据自身环境和需要选择。




---
# 1. 环境准备

执行下方脚本，检查 CANN Toolkit 是否可用，并把 CANN 环境变量加载到当前 notebook 进程。


In [ ]:
import os
import subprocess
import shlex
from pathlib import Path


def find_cann_home():
    candidates = []
    for key in ["ASCEND_HOME_PATH", "ASCEND_TOOLKIT_HOME"]:
        value = os.environ.get(key)
        if value:
            candidates.append(Path(value).expanduser())

    candidates.extend([
        Path.home() / "Ascend/cann",
        Path.home() / "Ascend/ascend-toolkit/latest",
        Path("/usr/local/Ascend/cann"),
        Path("/usr/local/Ascend/ascend-toolkit/latest"),
    ])

    for candidate in candidates:
        normalized = candidate
        if normalized.name in {"x86_64-linux", "aarch64-linux"}:
            normalized = normalized.parent
        set_env = normalized / "set_env.sh"
        if set_env.exists():
            return normalized.resolve(), set_env.resolve()

    raise RuntimeError("未找到 CANN Toolkit，请确认已安装 CANN，并设置环境变量。")


def source_cann_env(set_env):
    command = f"set -a && source {shlex.quote(str(set_env))} >/dev/null 2>&1 && env"
    result = subprocess.run(["bash", "-lc", command], check=True, text=True, capture_output=True)
    for line in result.stdout.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value


cann_home, cann_set_env = find_cann_home()
source_cann_env(cann_set_env)

print(f"CANN Toolkit: {cann_home}")


### 1.1 目录结构介绍

本课程代码按照如下目录结构存放：

```
├── src/04_02_static_tensor_cv_fusion
│   ├── scripts
│   │   ├── gen_data.py                         // 输入数据和 golden 数据生成脚本，支持 leakyrelu、mmad、mmad_leakyrelu
│   │   └── verify_result.py                    // 验证输出数据和 golden 数据是否一致的脚本
│   ├── CMakeLists.txt                          // 编译工程文件，定义 LeakyRelu、MMAD、MMAD+LeakyRelu 三个样例目标
│   ├── data_utils.h                            // Host 侧文件读入写出工具
│   ├── leakyrelu_basic_api.asc                 // 纯 Vector 静态 Tensor LeakyRelu Host 调用样例
│   ├── leakyrelu_operator.h                    // LeakyRelu 算子设备侧实现头文件
│   ├── mmad_basic_api.asc                      // 纯 Cube 静态 Tensor MMAD Host 调用样例
│   ├── mmad_operator.h                         // MMAD 算子设备侧实现头文件
│   ├── mmad_leakyrelu_basic_api.asc            // MMAD + LeakyRelu CV 融合 Host 调用样例
│   ├── mmad_leakyrelu_operator.h               // MMAD + LeakyRelu CV 融合算子设备侧实现头文件
│   ├── run_leakyrelu.sh                        // 静态 Tensor LeakyRelu 样例运行脚本
│   ├── run_mmad.sh                             // 静态 Tensor MMAD 样例运行脚本
│   └── run_mmad_leakyrelu.sh                   // MMAD + LeakyRelu CV 融合样例运行脚本
└── answer/04_02_static_tensor_cv_fusion
    ├── scripts
    │   └── gen_data.py                         // 课后实践 golden 数据生成脚本参考答案
    ├── mmad_leakyrelu_operator.h               // 课后实践算子设备侧修改参考答案
    └── mmad_leakyrelu_basic_api.asc            // 课后实践 Host 侧修改参考答案
```


---
# 2. 算子计算分析

静态 Tensor CV 融合算子开发的核心，不是记住某个样例的入口函数，而是理解一类融合算子的拆分方式：**Cube 侧负责矩阵类密集计算，Vector 侧负责逐元素后处理，两侧通过 GM 和同步事件完成数据交接**。

本节以 `MMAD + LeakyRelu` 作为承载样例，对应计算逻辑拆成两个清晰阶段：

```text
1. C = A * B + Bias           // MMAD（Cube 侧）
2. D = C > 0 ? C : C * alpha  // LeakyReLU（Vector 侧，alpha = 0.001）
```

- 第 1 阶段是 **MMAD**，由 Cube 核心完成矩阵乘与 Bias 累加。
- 第 2 阶段是 **LeakyRelu**，由 Vector 核心完成逐元素激活计算。

**MMAD（纯 Cube）**

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <tr><td rowspan="1" align="center">算子类型(OpType)</td><td colspan="4" align="center">MMAD(纯Cube)</td></tr>
  <tr><td rowspan="4" align="center">算子输入</td><td align="center">name</td><td align="center">shape</td><td align="center">data type</td><td align="center">format</td></tr>
  <tr><td align="center">矩阵A</td><td align="center">[M, K] = [512, 128]</td><td align="center">half</td><td align="center">ND</td></tr>
  <tr><td align="center">矩阵B</td><td align="center">[K, N] = [128, 128]</td><td align="center">half</td><td align="center">ND</td></tr>
  <tr><td align="center">Bias向量</td><td align="center">[N] = [128]</td><td align="center">half</td><td align="center">ND</td></tr>
  <tr><td rowspan="1" align="center">算子输出</td><td align="center">矩阵C</td><td align="center">[M, N] = [512, 128]</td><td align="center">half</td><td align="center">ND</td></tr>
  <tr><td rowspan="1" align="center">核函数名</td><td colspan="4" align="center">mmad_custom</td></tr>
  <tr><td rowspan="1" align="center">核数配置</td><td colspan="4" align="center">2 个 Cube 核，每个 Cube 核计算 [256, 128] 输出分块</td></tr>
</table>
<br>

**LeakyReLU（纯 Vector）**

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <tr><td rowspan="1" align="center">算子类型(OpType)</td><td colspan="4" align="center">LeakyReLU(纯Vector)</td></tr>
  <tr><td rowspan="2" align="center">算子输入</td><td align="center">name</td><td align="center">shape</td><td align="center">data type</td><td align="center">format</td></tr>
  <tr><td align="center">矩阵A</td><td align="center">[M, N] = [512, 128]</td><td align="center">half</td><td align="center">ND</td></tr>
  <tr><td rowspan="1" align="center">算子输出</td><td align="center">矩阵B</td><td align="center">[M, N] = [512, 128]</td><td align="center">half</td><td align="center">ND</td></tr>
  <tr><td rowspan="1" align="center">核函数名</td><td colspan="4" align="center">leakyrelu_custom</td></tr>
  <tr><td rowspan="1" align="center">核数配置</td><td colspan="4" align="center">4 个 Vector 核，每个 Cube 输出分块由 2 个 Vector 核协同处理</td></tr>
</table>




---
# 3. 静态 Tensor MMAD 算子

在 CV 融合样例中，输入矩阵需要先由 Cube 核完成 `MMAD(A, B) + Bias`，再交给后续 Vector 流程处理。本章先从一个纯 Cube 的 `mmad_custom` 算子开始。

## Kernel 入口

```cpp
template <uint32_t M, uint32_t K, uint32_t N, uint32_t singleCoreM, uint32_t baseM, uint32_t baseK, uint32_t baseN>
__global__ __cube__ void mmad_custom(__gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c)
```


<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">参数</th>
      <th align="left">含义</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>a</code></td>
      <td>输入矩阵 A 的 GM 地址</td>
    </tr>
    <tr>
      <td><code>b</code></td>
      <td>输入矩阵 B 的 GM 地址</td>
    </tr>
    <tr>
      <td><code>bias</code></td>
      <td>输入 Bias 向量的 GM 地址</td>
    </tr>
    <tr>
      <td><code>c</code></td>
      <td>输出矩阵 C 的 GM 地址</td>
    </tr>
  </tbody>
</table>

样例规格如下：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">项目</th>
      <th align="left">取值</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>A 矩阵</td>
      <td><code>[M, K] = [512, 128]</code>, <code>half</code>, ND</td>
    </tr>
    <tr>
      <td>B 矩阵</td>
      <td><code>[K, N] = [128, 128]</code>, <code>half</code>, ND</td>
    </tr>
    <tr>
      <td>Bias 向量</td>
      <td><code>[N] = [128]</code>, <code>half</code>, ND</td>
    </tr>
    <tr>
      <td>输出 C</td>
      <td><code>[M, N] = [512, 128]</code>, <code>half</code>, ND</td>
    </tr>
    <tr>
      <td>Cube 核数</td>
      <td>2</td>
    </tr>
    <tr>
      <td>单 Cube 核 M 方向计算量</td>
      <td><code>singleCoreM = 256</code></td>
    </tr>
  </tbody>
</table>

静态 Tensor 写法直接面向硬件存储分配 `LocalTensor`：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">存储位置</th>
      <th align="left">样例中的对象</th>
      <th align="left">作用</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>L1</td>
      <td><code>a1Local</code>, <code>b1Local</code>, <code>bias1Local</code></td>
      <td>接收 GM 搬入后的 A/B 分块和 Bias 向量</td>
    </tr>
    <tr>
      <td>L0A</td>
      <td><code>a2Local</code></td>
      <td>Cube 计算读取的 A 分形数据</td>
    </tr>
    <tr>
      <td>L0B</td>
      <td><code>b2Local</code></td>
      <td>Cube 计算读取的 B 分形数据</td>
    </tr>
    <tr>
      <td>BIAS</td>
      <td><code>bias2Local</code></td>
      <td>MMAD 原生 Bias 输入，承载 half → float 后的 Bias 向量</td>
    </tr>
    <tr>
      <td>L0C</td>
      <td><code>cLocal</code></td>
      <td><code>Mmad</code> 输出累加结果</td>
    </tr>
  </tbody>
</table>

核心数据流如下：

```text
GM(A/B/Bias) → L1(A1/B1/Bias1) → L0A/L0B/BIAS(A2/B2/Bias2) → L0C → GM(C)
```


## 3.1 Mmad 算子代码实现

让我们一起在 `src/04_02_static_tensor_cv_fusion/mmad_operator.h` 这个文件中完成 Mmad 算子实现。

### 3.1.1 框架搭建

第一步先写出框架代码包括常量初始化、`KernelMmad` 模板类、`Init`/`Process` 成员函数以及纯 Cube kernel 入口。


In [ ]:
%%writefile src/04_02_static_tensor_cv_fusion/mmad_operator.h
/* !
 * \file mmad_operator.h
 * \brief 使用静态 Tensor 编程模式实现带 Bias 的 MMAD 单算子。
 */

#ifndef ASCENDC_04_02_MMAD_OPERATOR_H
#define ASCENDC_04_02_MMAD_OPERATOR_H

#include "kernel_operator.h"

constexpr uint32_t CUBE_BLOCK = 16;

template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t singleCoreM, uint32_t baseM, uint32_t baseK, uint32_t baseN>
class KernelMmad {
public:
    __aicore__ inline void Init(__gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c)
    {
        // TODO(MMAD_INIT_GLOBAL_TENSOR)
    }

    __aicore__ inline void Process()
    {
        // TODO(MMAD_ALLOC_LOCAL_TENSOR)
        // TODO(MMAD_PREPARE_FRACTAL)
        // TODO(MMAD_COMPUTE_AND_STORE)
    }
private:
    AscendC::GlobalTensor<half> aGM;
    AscendC::GlobalTensor<half> bGM;
    AscendC::GlobalTensor<half> biasGM;
    AscendC::GlobalTensor<half> cGM;
};

template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t singleCoreM, uint32_t baseM, uint32_t baseK, uint32_t baseN>
__global__ __cube__ void mmad_custom(__gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c)
{
    AscendC::InitSocState();
    KernelMmad<M, K, N, singleCoreM, baseM, baseK, baseN> op;
    op.Init(a, b, bias, c);
    op.Process();
    AscendC::PipeBarrier<PIPE_ALL>();
}

#endif // ASCENDC_04_02_MMAD_OPERATOR_H


### 3.1.2 矩阵计算划分

MMAD 的输出矩阵为 `[M, N] = [512, 128]`，样例启动 2 个 Cube 核，因此每个 Cube 核负责 `singleCoreM = 256` 行输出：

```text
Cube 0: A[0:256, :]   × B[:, :] → C[0:256, :]
Cube 1: A[256:512, :] × B[:, :] → C[256:512, :]
```

`GetBlockIdx()` 用于获得当前 Cube 核编号。由于每个核只沿 M 方向切分，B 矩阵和 Bias 向量对所有 Cube 核相同；A 和 C 需要根据 `mIterIdx` 加上当前核负责的起始偏移。

![alt text](images/04_02_static_tensor_cv_fusion/3_1_1_mmad_tiling.svg)


In [ ]:
from pathlib import Path

path = Path("src/04_02_static_tensor_cv_fusion/mmad_operator.h")
text = path.read_text(encoding="utf-8")
marker = "        // TODO(MMAD_INIT_GLOBAL_TENSOR)"
replacement = """        // 1. 矩阵计算划分：根据当前 Cube 核编号确定本核负责的 M 方向分块。
        // A 和 C 沿 M 方向切分，因此需要加上当前分块偏移；B 和 Bias 被所有 Cube 核复用。
        uint32_t mIterIdx = AscendC::GetBlockIdx() % (M / singleCoreM);
        aGM.SetGlobalBuffer((__gm__ half*)a + mIterIdx * singleCoreM * K);
        bGM.SetGlobalBuffer((__gm__ half*)b);
        biasGM.SetGlobalBuffer((__gm__ half*)bias);
        cGM.SetGlobalBuffer((__gm__ half*)c + mIterIdx * singleCoreM * N);"""

if marker not in text:
    raise RuntimeError(f"未找到插入标记: {marker}")
path.write_text(text.replace(marker, replacement, 1), encoding="utf-8")
print(f"已插入矩阵计算划分代码: {path}")


### 3.1.3 静态 Tensor 初始化

第二步在 `Process` 中申请各级硬件存储上的静态 Tensor。这里使用 `LocalMemAllocator`，它会在对应硬件存储空间中按申请顺序分配地址。

本样例需要：

- L1：`a1Local`、`b1Local`、`bias1Local`，承接 GM 搬入的数据；
- L0A/L0B：`a2Local`、`b2Local`，承接 Cube 计算需要的分形矩阵；
- BIAS：`bias2Local`，作为 `Mmad` 的原生 Bias 输入；
- L0C：`cLocal`，承接 `Mmad` 的 float 累加结果。


In [ ]:
from pathlib import Path

path = Path("src/04_02_static_tensor_cv_fusion/mmad_operator.h")
text = path.read_text(encoding="utf-8")
marker = "        // TODO(MMAD_ALLOC_LOCAL_TENSOR)"
replacement = """        // 2. 静态 Tensor 初始化：为 MMAD 计算涉及的各级硬件存储创建分配器。
        // L1 承接 GM 搬入数据，L0A/L0B 承接分形矩阵，BIAS 承接原生 Bias，L0C 承接累加结果。
        AscendC::LocalMemAllocator<AscendC::Hardware::L1> l1Allocator;
        AscendC::LocalMemAllocator<AscendC::Hardware::L0A> l0aAllocator;
        AscendC::LocalMemAllocator<AscendC::Hardware::L0B> l0bAllocator;
        AscendC::LocalMemAllocator<AscendC::Hardware::L0C> l0cAllocator;
        AscendC::LocalMemAllocator<AscendC::Hardware::BIAS> biasAllocator;

        // 3. 静态 Tensor 初始化：按数据流申请 L1、L0A、L0B、BIAS 和 L0C 上的 LocalTensor。
        // 这些 Tensor 的大小与当前 tile 的 baseM/baseK/baseN 相关，编译期即可确定。
        AscendC::LocalTensor<half> a1Local = l1Allocator.Alloc<half>(baseM * baseK);
        AscendC::LocalTensor<half> b1Local = l1Allocator.Alloc<half>(baseK * baseN);
        AscendC::LocalTensor<half> bias1Local = l1Allocator.Alloc<half>(baseN);
        AscendC::LocalTensor<half> a2Local = l0aAllocator.Alloc<half>(baseM * baseK);
        AscendC::LocalTensor<half> b2Local = l0bAllocator.Alloc<half>(baseK * baseN);
        AscendC::LocalTensor<float> bias2Local = biasAllocator.Alloc<float>(baseN);
        AscendC::LocalTensor<float> cLocal = l0cAllocator.Alloc<float>(baseM * baseN);"""

if marker not in text:
    raise RuntimeError(f"未找到插入标记: {marker}")
path.write_text(text.replace(marker, replacement, 1), encoding="utf-8")
print(f"已插入静态 Tensor 初始化代码: {path}")


### 3.1.4 准备 MMAD 计算所需要的分形矩阵

第三步完成 MMAD 计算前的数据准备。输入文件中的 A/B/Bias 都是 ND 视角的数据，Cube 计算不能直接从 GM 读取这些 ND 数据，而是需要先搬入 L1，再通过 `LoadData` 转换到 L0A/L0B/BT 的分形布局。

本阶段包含三个关键点：

1. `DataCopy` 把 A、B、Bias 从 GM 搬到 L1；
2. `SetFlag/WaitFlag<MTE2_MTE1>` 保证 L1 数据已由 MTE2 搬完，MTE1 才开始 `LoadData`；
3. `LoadData` 把 A/B 从 L1 搬到 L0A/L0B，同时把 Bias 转成 float 后放入 BIAS 空间。

不同 NPU 架构的 `LoadData` 参数形态略有差异，因此这里保留 `__NPU_ARCH__ == 2201` 与 `__NPU_ARCH__ == 3510` 两套写法。


In [ ]:
from pathlib import Path

path = Path("src/04_02_static_tensor_cv_fusion/mmad_operator.h")
text = path.read_text(encoding="utf-8")
marker = "        // TODO(MMAD_PREPARE_FRACTAL)"
replacement = """        // 4. 准备分形矩阵：先把 A、B 和 Bias 从 GM 搬入 L1。
        // A/B 使用 Nd2NzParams 描述 ND 到 NZ 的搬运形状，Bias 是长度为 baseN 的一维向量。
        AscendC::DataCopy(a1Local, aGM, AscendC::Nd2NzParams{1, baseM, baseK, 0, K, baseM, 1, 0});
        AscendC::DataCopy(b1Local, bGM, AscendC::Nd2NzParams{1, baseK, baseN, 0, N, baseK, 1, 0});
        AscendC::DataCopy(bias1Local, biasGM, baseN);

        // 5. 准备分形矩阵：等待 GM->L1 的 MTE2 搬运完成，再允许 MTE1 读取 L1。
        // 该同步保护后续 LoadData 不会读到尚未搬运完成的 A/B/Bias 数据。
        AscendC::SetFlag<AscendC::HardEvent::MTE2_MTE1>(EVENT_ID0);
        AscendC::WaitFlag<AscendC::HardEvent::MTE2_MTE1>(EVENT_ID0);

        // 6. 准备分形矩阵：将 Bias 从 L1 转换并搬入 BIAS/C2 空间，作为 Mmad 原生 Bias 输入。
        AscendC::DataCopy(bias2Local, bias1Local, {1, static_cast<uint16_t>(baseN * sizeof(float) / 64), 0, 0});
#if defined(__NPU_ARCH__) && (__NPU_ARCH__ == 2201)
        // 7. 准备分形矩阵：dav-2201 上分行把 A 从 L1 搬到 L0A，并转换为 Cube 读取布局。
        for (int i = 0; i < baseM / CUBE_BLOCK; ++i) {
            AscendC::LoadData(a2Local[i * baseK * CUBE_BLOCK], a1Local[i * 512 / sizeof(half)],
                AscendC::LoadData2DParams{0, baseK / CUBE_BLOCK, baseM / CUBE_BLOCK, 0, 0, false, 0});
        }
        // 8. 准备分形矩阵：dav-2201 上分列把 B 从 L1 搬到 L0B，并按 MMAD 需要转置布局。
        for (int i = 0; i < baseK / CUBE_BLOCK; ++i) {
            AscendC::LoadData(b2Local[i * baseN * CUBE_BLOCK], b1Local[i * 512 / sizeof(half)],
                AscendC::LoadData2DParams{0, baseN / CUBE_BLOCK, baseK / CUBE_BLOCK, 0, 0, true, 0});
        }
#elif defined(__NPU_ARCH__) && (__NPU_ARCH__ == 3510)
        // 9. 准备分形矩阵：dav-3510 使用 V2 参数一次性描述 A 的 L1->L0A 分形搬运。
        AscendC::LoadData(a2Local, a1Local,
            AscendC::LoadData2DParamsV2{0, 0, baseM / CUBE_BLOCK, baseK / CUBE_BLOCK,
                baseM / CUBE_BLOCK, baseM / CUBE_BLOCK, false, 0});
        // 10. 准备分形矩阵：dav-3510 使用 V2 参数一次性描述 B 的 L1->L0B 分形搬运。
        AscendC::LoadData(b2Local, b1Local,
            AscendC::LoadData2DParamsV2{0, 0, baseK / CUBE_BLOCK, baseN / CUBE_BLOCK,
                baseK / CUBE_BLOCK, baseN / CUBE_BLOCK, true, 0});
#endif

        // 11. 准备分形矩阵：等待 L1->L0A/L0B 的 MTE1 搬运完成，再允许 M 流水执行 Mmad。
        AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(EVENT_ID0);
        AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(EVENT_ID0);"""

if marker not in text:
    raise RuntimeError(f"未找到插入标记: {marker}")
path.write_text(text.replace(marker, replacement, 1), encoding="utf-8")
print(f"已插入分形矩阵准备代码: {path}")


### 3.1.5 数据搬出

最后一步执行 `Mmad` 并写回结果。`Mmad` 从 L0A/L0B/BIAS 读取输入，计算结果先落在 L0C；随后 `Fixpipe` 从 L0C 读取结果并写回 GM。

这里有两个同步点需要特别注意：

- `MTE1_M`：保证 `LoadData` 已经准备好 L0A/L0B，`Mmad` 才能读取；
- `M_FIX`：保证 `Mmad` 已经写好 L0C，`Fixpipe` 才能读取并写回 GM。

`FixpipeParamsV220` 中同时描述了 L0C→GM 的形状、源/目的 stride，以及 float32 到 float16 的转换方式。


In [ ]:
from pathlib import Path

path = Path("src/04_02_static_tensor_cv_fusion/mmad_operator.h")
text = path.read_text(encoding="utf-8")
marker = "        // TODO(MMAD_COMPUTE_AND_STORE)"
replacement = """        // 12. 数据搬出：配置并执行 Mmad，输入来自 L0A/L0B/BIAS，输出累加到 L0C。
        // cmatrixInitVal=false 表示 C 矩阵不额外清零初始化，本 tile 由当前 Mmad 直接生成。
        AscendC::MmadParams mmADParams;
        mmADParams.m = baseM;
        mmADParams.n = baseN;
        mmADParams.k = baseK;
        mmADParams.cmatrixInitVal = false;
        AscendC::Mmad(cLocal, a2Local, b2Local, bias2Local, mmADParams);

        // 13. 数据搬出：等待 M 流水把结果写入 L0C 后，再允许 FIX 流水读取并写回 GM。
        AscendC::SetFlag<AscendC::HardEvent::M_FIX>(EVENT_ID0);
        AscendC::WaitFlag<AscendC::HardEvent::M_FIX>(EVENT_ID0);

        // 14. 数据搬出：Fixpipe 将 L0C 中的 float 累加结果转换为 half，并写回当前 Cube 核负责的 C 分块。
        AscendC::Fixpipe(cGM, cLocal,
            AscendC::FixpipeParamsV220{baseN, baseM, baseM, N, false, QuantMode_t::F322F16, 0, 1, 0, 0, 0});"""

if marker not in text:
    raise RuntimeError(f"未找到插入标记: {marker}")
path.write_text(text.replace(marker, replacement, 1), encoding="utf-8")
print(f"已插入 MMAD 计算和数据搬出代码: {path}")


## 3.2 MMAD 样例的时序同步分析

流水同步核心是保证每个指令依赖的数据已经准备完毕：

![alt text](images/04_02_static_tensor_cv_fusion/3_2_mmad_pipeline.svg)

对应到代码，三个同步点分别保护三段真实的数据依赖：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">同步点</th>
      <th align="left">指令顺序</th>
      <th align="left">保护的数据</th>
      <th align="left">如果缺失，可能发生什么</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>MTE2_MTE1</code></td>
      <td><code>DataCopy</code> → <code>LoadData</code></td>
      <td>L1 中的 A/B 分块</td>
      <td><code>LoadData</code> 可能从 L1 读取到尚未搬完的数据</td>
    </tr>
    <tr>
      <td><code>MTE1_M</code></td>
      <td><code>LoadData</code> → <code>Mmad</code></td>
      <td>L0A/L0B 中的分形数据</td>
      <td><code>Mmad</code> 可能提前读取 L0A/L0B</td>
    </tr>
    <tr>
      <td><code>M_FIX</code></td>
      <td><code>Mmad</code> → <code>Fixpipe</code></td>
      <td>L0C 中的矩阵乘结果</td>
      <td><code>Fixpipe</code> 可能把未完成的 L0C 写回 GM</td>
    </tr>
  </tbody>
</table>


## 3.3 运行 MMAD 单算子样例

执行该单元，完成纯 Cube MMAD 单算子的输入生成、编译、运行和精度校验。该样例通过 `mmad_basic_api.asc` 调用 `mmad_operator.h` 中的 `mmad_custom`。

In [ ]:
!cd src/04_02_static_tensor_cv_fusion && bash run_mmad.sh --mode=npu --npu-arch=dav-2201


---
# 4. 静态 Tensor LeakyRelu 算子

在 CV 融合样例中，MMAD 结果需要由 AIV/Vector 侧执行逐元素后处理。本节先实现一个纯Vector的`leakyrelu_custom`算子。

### Kernel 入口

```cpp
template <uint32_t totalLength, uint32_t tileLength>
__global__ __vector__ void leakyrelu_custom(__gm__ uint8_t* x, __gm__ uint8_t* y)
```

使用 `__vector__` 声明，表示该样例只在 Vector 侧运行。Host 侧负责读取输入向量、申请 Device 内存、启动 kernel，并将输出写回文件。

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">参数</th>
      <th align="left">含义</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>x</code></td>
      <td>输入向量的 GM 地址</td>
    </tr>
    <tr>
      <td><code>y</code></td>
      <td>输出向量的 GM 地址</td>
    </tr>
  </tbody>
</table>

样例规格如下：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">项目</th>
      <th align="left">取值</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>输入 X</td>
      <td><code>[512, 128]</code>, <code>half</code>, ND</td>
    </tr>
    <tr>
      <td>输出 Y</td>
      <td><code>[512, 128]</code>, <code>half</code>, ND</td>
    </tr>
    <tr>
      <td>AIV 核数</td>
      <td>8</td>
    </tr>
    <tr>
      <td>单 Vec 核 M 方向计算量</td>
      <td><code>singleCoreM = 128</code></td>
    </tr>
    <tr>
      <td>激活斜率</td>
      <td><code>negativeSlope = 0.001</code></td>
    </tr>
  </tbody>
</table>

静态 Tensor 写法直接从 UB 分配 `LocalTensor`：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">阶段</th>
      <th align="left">关键接口</th>
      <th align="left">数据流</th>
      <th align="left">同步点</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>输入搬入</td>
      <td><code>DataCopy</code></td>
      <td>GM → UB</td>
      <td><code>MTE2_V</code></td>
    </tr>
    <tr>
      <td>向量计算</td>
      <td><code>LeakyRelu</code></td>
      <td>UB → UB</td>
      <td><code>V_MTE3</code></td>
    </tr>
    <tr>
      <td>结果写回</td>
      <td><code>DataCopy</code></td>
      <td>UB → GM</td>
      <td>写回前等待 <code>V_MTE3</code></td>
    </tr>
  </tbody>
</table>

核心数据流如下：

```text
GM(X) → UB(X) → Vector → UB(Y) → GM(Y)
```





### 4.1 LeakyRelu 算子代码实现

让我们一起在 `src/04_02_static_tensor_cv_fusion/leakyrelu_operator.h` 这个文件中完成 LeakyRelu 算子实现。


#### 4.1.1 框架搭建

先写出头文件、`KernelLeakyRelu` 类和 `leakyrelu_custom` 入口。


In [ ]:
%%writefile src/04_02_static_tensor_cv_fusion/leakyrelu_operator.h
/* !
 * \file leakyrelu_operator.h
 * \brief 使用静态 Tensor 编程模式实现 LeakyRelu 向量算子。
 */

#ifndef ASCENDC_04_02_LEAKYRELU_OPERATOR_H
#define ASCENDC_04_02_LEAKYRELU_OPERATOR_H

#include "kernel_operator.h"

template <uint32_t totalLength, uint32_t tileLength>
class KernelLeakyRelu {
public:
    __aicore__ inline void Init(__gm__ uint8_t* x, __gm__ uint8_t* y, uint32_t blockOffset = AscendC::GetBlockIdx() * tileLength)
    {
        // TODO(LEAKYRELU_INIT_GLOBAL_TENSOR)
    }

    __aicore__ inline void Process()
    {
        // TODO(LEAKYRELU_ALLOC_LOCAL_TENSOR)
        // TODO(LEAKYRELU_COMPUTE_AND_STORE)
    }

private:
    AscendC::GlobalTensor<half> xGM;
    AscendC::GlobalTensor<half> yGM;
};

template <uint32_t totalLength, uint32_t tileLength>
__global__ __vector__ void leakyrelu_custom(__gm__ uint8_t* x, __gm__ uint8_t* y)
{
    AscendC::InitSocState();
    KernelLeakyRelu<totalLength, tileLength> op;
    op.Init(x, y);
    op.Process();
    AscendC::PipeBarrier<PIPE_ALL>();
}

#endif // ASCENDC_04_02_LEAKYRELU_OPERATOR_H


#### 4.1.2 地址初始化

根据当前 Vector 核负责的 `blockOffset`，绑定输入和输出 GM 分片。


In [ ]:
from pathlib import Path

path = Path("src/04_02_static_tensor_cv_fusion/leakyrelu_operator.h")
text = path.read_text(encoding="utf-8")
marker = "        // TODO(LEAKYRELU_INIT_GLOBAL_TENSOR)"
replacement = """        // 1. 地址初始化：根据当前 Vector 核负责的 blockOffset 绑定输入和输出 GM 分片。
        // 纯 LeakyRelu 默认按 GetBlockIdx() * tileLength 切分；融合场景可传入自定义 blockOffset 复用该类。
        xGM.SetGlobalBuffer((__gm__ half*)x + blockOffset);
        yGM.SetGlobalBuffer((__gm__ half*)y + blockOffset);"""

if marker not in text:
    raise RuntimeError(f"未找到插入标记: {marker}")
path.write_text(text.replace(marker, replacement, 1), encoding="utf-8")
print(f"已插入地址初始化代码: {path}")


#### 4.1.3 静态 Tensor 初始化

在 UB 上申请当前核处理的 `xLocal`。


In [ ]:
from pathlib import Path

path = Path("src/04_02_static_tensor_cv_fusion/leakyrelu_operator.h")
text = path.read_text(encoding="utf-8")
marker = "        // TODO(LEAKYRELU_ALLOC_LOCAL_TENSOR)"
replacement = """        // 2. 静态 Tensor 初始化：在 UB 上申请当前 Vector 核处理的 tile 缓冲区。
        // xLocal 既保存搬入的输入数据，也承载 LeakyRelu 原地计算后的输出数据。
        AscendC::LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
        AscendC::LocalTensor<half> xLocal = ubAllocator.Alloc<half>(tileLength);"""

if marker not in text:
    raise RuntimeError(f"未找到插入标记: {marker}")
path.write_text(text.replace(marker, replacement, 1), encoding="utf-8")
print(f"已插入静态 Tensor 初始化代码: {path}")


#### 4.1.4 输入搬入、计算和数据搬出

完成 `GM -> UB -> Vector -> GM` 的 LeakyRelu 主流程。


In [ ]:
from pathlib import Path

path = Path("src/04_02_static_tensor_cv_fusion/leakyrelu_operator.h")
text = path.read_text(encoding="utf-8")
marker = "        // TODO(LEAKYRELU_COMPUTE_AND_STORE)"
replacement = """        // 3. 输入搬入与 Vector 计算：先把当前 tile 从 GM 搬入 UB。
        // DataCopy 由 MTE2 流水执行，后续 Vector 计算必须等待搬入完成。
        AscendC::DataCopy(xLocal, xGM, tileLength);
        AscendC::SetFlag<AscendC::HardEvent::MTE2_V>(EVENT_ID0);
        AscendC::WaitFlag<AscendC::HardEvent::MTE2_V>(EVENT_ID0);

        // 4. 输入搬入与 Vector 计算：在 UB 上执行原地 LeakyRelu，负半轴斜率为 0.001。
        AscendC::LeakyRelu(xLocal, xLocal, (half)0.001, tileLength);

        // 5. 数据搬出：等待 Vector 计算完成，再允许 MTE3 从 UB 读取结果并写回 GM。
        AscendC::SetFlag<AscendC::HardEvent::V_MTE3>(EVENT_ID0);
        AscendC::WaitFlag<AscendC::HardEvent::V_MTE3>(EVENT_ID0);

        // 6. 数据搬出：将当前 Vector 核处理完成的 tile 写回输出 GM 分片。
        AscendC::DataCopy(yGM, xLocal, tileLength);"""

if marker not in text:
    raise RuntimeError(f"未找到插入标记: {marker}")
path.write_text(text.replace(marker, replacement, 1), encoding="utf-8")
print(f"已插入输入搬入、计算和数据搬出代码: {path}")


### 4.2 LeakyRelu 样例的时序同步分析

纯 LeakyRelu 的数据只在 AIV 侧流动，核心是保证 UB 中的数据在 Vector 计算前已经搬入，并在写回 GM 前已经完成计算：

![alt text](images/04_02_static_tensor_cv_fusion/4_2_leakyrelu_pipeline.svg)

对应到代码，两个同步点分别保护以下依赖：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">同步点</th>
      <th align="left">指令顺序</th>
      <th align="left">保护的数据</th>
      <th align="left">如果缺失，可能发生什么</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>MTE2_V</code></td>
      <td><code>DataCopy</code> → <code>LeakyRelu</code></td>
      <td>UB 中的输入向量分片</td>
      <td>Vector 可能读取到尚未搬入完成的数据</td>
    </tr>
    <tr>
      <td><code>V_MTE3</code></td>
      <td><code>LeakyRelu</code> → <code>DataCopy</code></td>
      <td>UB 中的激活结果</td>
      <td>MTE3 可能提前写回未完成计算的数据</td>
    </tr>
  </tbody>
</table>






### 4.3 运行 LeakyRelu 单算子样例

执行该单元，完成纯 Vector LeakyRelu 单算子的输入生成、编译、运行和精度校验。该样例通过 `leakyrelu_basic_api.asc` 调用 `leakyrelu_operator.h` 中的 `leakyrelu_custom`。





In [ ]:
!cd src/04_02_static_tensor_cv_fusion && bash run_leakyrelu.sh --mode=npu --npu-arch=dav-2201


---
# 5. CV 融合算子实现

在前两章中，MMAD 单算子只使用 AIC/Cube 核完成矩阵计算，LeakyRelu 单算子只使用 AIV/Vector 核完成逐元素计算。本章要实现的 `MMAD + LeakyRelu` 是一个 Mix 算子：AIC 侧先完成 `MMAD + Bias`，AIV 侧再读取 AIC 产生的中间结果并执行 LeakyRelu。

## 5.1 核间同步概念

核间同步用于解决“一个核要读取的数据依赖另一个核的计算结果”这一类时序问题。不同核的执行是并行的，代码中 AIC 分支写在 AIV 分支前，并不代表 AIV 一定会等到 AIC 完成。只要 AIV 读取的数据由 AIC 生产，就需要显式建立核间同步关系。

本样例的跨核数据流如下：

```text
AIC：MMAD(A, B) + Bias → Fixpipe 写回 GM 中的 C_tile
AIV：等待 AIC 通知 → 从 GM 读取 C_tile → LeakyRelu → 写回 GM
```

![MMAD + LeakyRelu 中的核间同步](images/04_02_static_tensor_cv_fusion/5_0_cross_core_sync.svg)

`__mix__(1, 2)` 表示一个 group 内由 1 个 AIC block 搭配 2 个 AIV subblock。AIC 负责当前输出矩阵块的 Cube 计算；两个 AIV 分别处理该块的上半段和下半段 Vector 后处理。AIV 读取的是 AIC 通过 `Fixpipe` 写回 GM 的中间结果，因此必须等 AIC 写回完成后再执行 `DataCopy(GM → UB)`。

本课程使用 `CrossCoreSetFlag` / `CrossCoreWaitFlag` 表达这段依赖：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">接口</th>
      <th align="left">调用位置</th>
      <th align="left">作用</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>CrossCoreSetFlag&lt;0x2, PIPE_FIX&gt;(0)</code></td>
      <td>AIC 分支，<code>Fixpipe</code> 写回 GM 之后</td>
      <td>通知同 group 内关联 AIV：AIC 侧结果块已经写回</td>
    </tr>
    <tr>
      <td><code>CrossCoreWaitFlag(0)</code></td>
      <td>AIV 分支，读取 GM 中间结果之前</td>
      <td>阻塞 AIV 后续指令，直到收到 AIC 通知</td>
    </tr>
  </tbody>
</table>

如果缺少这组核间同步，AIV 可能在 AIC 的 `Fixpipe` 尚未完成时读取 GM，后续 LeakyRelu 处理的就是未完成或上一轮遗留的数据。

## 5.2 代码实现入口

让我们一起在 `src/04_02_static_tensor_cv_fusion/mmad_leakyrelu_operator.h` 这个文件中完成 MMAD + LeakyRelu CV 融合算子实现。


### 5.2.1 框架搭建

先写出融合算子头文件和 `mmad_leakyrelu_custom` 混合核入口。

In [ ]:
%%writefile src/04_02_static_tensor_cv_fusion/mmad_leakyrelu_operator.h
/* !
 * \file mmad_leakyrelu_operator.h
 * \brief 使用静态 Tensor 编程模式实现 MMAD + LeakyRelu CV 融合算子。
 */

#ifndef ASCENDC_04_02_MMAD_LEAKYRELU_OPERATOR_H
#define ASCENDC_04_02_MMAD_LEAKYRELU_OPERATOR_H

#include "kernel_operator.h"
#include "mmad_operator.h"
#include "leakyrelu_operator.h"

template <
    uint32_t M, uint32_t K, uint32_t N, uint32_t singleCoreM, uint32_t singleCoreK, uint32_t singleCoreN,
    uint32_t baseM, uint32_t baseK, uint32_t baseN>
__global__ __mix__(1, 2) void mmad_leakyrelu_custom(
    __gm__ uint8_t* a, __gm__ uint8_t* b, __gm__ uint8_t* bias, __gm__ uint8_t* c)
{
    AscendC::InitSocState();
    if ASCEND_IS_AIC {
        // TODO(MMAD_LEAKYRELU_AIC_PROCESS)
    }
    if ASCEND_IS_AIV {
        // TODO(MMAD_LEAKYRELU_AIV_PROCESS)
    }
    AscendC::PipeBarrier<PIPE_ALL>();
}

#endif // ASCENDC_04_02_MMAD_LEAKYRELU_OPERATOR_H


### 5.2.2 AIC 侧 MMAD 计算

AIC 侧复用 `KernelMmad` 完成矩阵乘计算，并在结果写回 GM 后通知 AIV。

In [ ]:
from pathlib import Path

path = Path("src/04_02_static_tensor_cv_fusion/mmad_leakyrelu_operator.h")
text = path.read_text(encoding="utf-8")
marker = "        // TODO(MMAD_LEAKYRELU_AIC_PROCESS)"
replacement = """        // 1. AIC 侧 MMAD 计算：复用 KernelMmad 完成当前 Cube 核负责的输出分块。
        KernelMmad<M, K, N, singleCoreM, baseM, baseK, baseN> op;
        op.Init(a, b, bias, c);
        op.Process();

        // 2. AIC/AIV 跨核同步：Fixpipe 写回 GM 后，通知对应 Vector 核可以读取结果块。
        AscendC::CrossCoreSetFlag<0x2, PIPE_FIX>(0);"""

if marker not in text:
    raise RuntimeError(f"未找到插入标记: {marker}")
path.write_text(text.replace(marker, replacement, 1), encoding="utf-8")
print(f"已插入 AIC 侧 MMAD 计算代码: {path}")


### 5.2.3 AIV 侧 LeakyRelu 后处理

AIV 侧等待 AIC 通知后，从 GM 读取对应半块结果，在 UB 上执行 LeakyRelu 并写回 GM。

In [ ]:
from pathlib import Path

path = Path("src/04_02_static_tensor_cv_fusion/mmad_leakyrelu_operator.h")
text = path.read_text(encoding="utf-8")
marker = "        // TODO(MMAD_LEAKYRELU_AIV_PROCESS)"
replacement = """        // 3. AIV 侧 LeakyRelu 后处理：每 2 个 Vector 核对应 1 个 Cube 核。
        // 两个 Vector 核分别处理当前 Cube 输出块的上半段和下半段。
        KernelLeakyRelu<M * N, baseM / 2 * baseN> op;
        uint32_t blockOffset = AscendC::GetBlockIdx() / 2 * singleCoreM * N +
            AscendC::GetBlockIdx() % 2 * (baseM / 2 * N);
        op.Init(c, c, blockOffset);

        // 4. AIC/AIV 跨核同步：等待 AIC 侧 Fixpipe 写回当前结果块后再读取 GM。
        AscendC::CrossCoreWaitFlag(0);
        op.Process();"""

if marker not in text:
    raise RuntimeError(f"未找到插入标记: {marker}")
path.write_text(text.replace(marker, replacement, 1), encoding="utf-8")
print(f"已插入 AIV 侧 LeakyRelu 后处理代码: {path}")


### 5.3 CV 融合样例的时序同步分析

融合样例由 AIC 侧 MMAD、AIC/AIV 跨核同步和 AIV 侧 LeakyRelu 后处理组成：

![alt text](images/04_02_static_tensor_cv_fusion/5_2_mmad_leakyrelu_pipeline.svg)

对应到代码，关键同步点分别保护以下依赖：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">同步点</th>
      <th align="left">指令顺序</th>
      <th align="left">保护的数据</th>
      <th align="left">如果缺失，可能发生什么</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>MTE2_MTE1</code></td>
      <td><code>DataCopy</code> → <code>LoadData</code></td>
      <td>L1 中的 A/B 分块</td>
      <td><code>LoadData</code> 可能从 L1 读取到尚未搬完的数据</td>
    </tr>
    <tr>
      <td><code>MTE1_M</code></td>
      <td><code>LoadData</code> → <code>Mmad</code></td>
      <td>L0A/L0B 中的分形数据</td>
      <td><code>Mmad</code> 可能提前读取 L0A/L0B</td>
    </tr>
    <tr>
      <td><code>M_FIX</code></td>
      <td><code>Mmad</code> → <code>Fixpipe</code></td>
      <td>L0C 中的矩阵乘结果</td>
      <td><code>Fixpipe</code> 可能把未完成的 L0C 写回 GM</td>
    </tr>
    <tr>
      <td><code>CrossCoreSetFlag/WaitFlag</code></td>
      <td>AIC <code>Fixpipe</code> → AIV <code>DataCopy</code></td>
      <td>GM 中的 C_tile</td>
      <td>AIV 可能读取到尚未写回的结果块</td>
    </tr>
    <tr>
      <td><code>MTE2_V</code></td>
      <td><code>DataCopy</code> → <code>LeakyRelu</code></td>
      <td>UB 中的 C_tile</td>
      <td>Vector 可能读取到尚未搬入完成的数据</td>
    </tr>
    <tr>
      <td><code>V_MTE3</code></td>
      <td><code>LeakyRelu</code> → <code>DataCopy</code></td>
      <td>UB 中的激活结果</td>
      <td>MTE3 可能提前写回未完成计算的数据</td>
    </tr>
  </tbody>
</table>

### 5.4 运行 CV 融合样例

执行下方代码块，按 NPU 模式完成 `MMAD + LeakyRelu` CV 融合样例的输入生成、编译、运行和精度校验。

In [ ]:
!cd src/04_02_static_tensor_cv_fusion && bash run_mmad_leakyrelu.sh --mode=npu --npu-arch=dav-2201


### 5.5 CV 融合算子性能

CV 融合算子开发的必要性，最直观地体现在端到端执行耗时上。下面对比同一组 `MMAD + LeakyRelu` 计算分别采用融合实现和串行调用两个单算子时的 Profiling 结果。

**融合算子执行耗时：**

![CV 融合算子执行耗时](images/04_02_static_tensor_cv_fusion/5_5_cv_fusion_op_cost.jpg)

**算子串行执行耗时：**

![MMAD 与 LeakyRelu 串行执行耗时](images/04_02_static_tensor_cv_fusion/5_5_cv_serial_op_cost.jpg)

从 Timeline 的 Slice List 可以看到：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">实现方式</th>
      <th align="left">总 Wall Duration</th>
      <th align="left">Kernel 计算耗时</th>
      <th align="left">Launch 次数</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>CV 融合算子</td>
      <td>约 <code>1.743690 ms</code></td>
      <td><code>KERNEL_MIX_AIC</code> 约 <code>0.011005 ms</code></td>
      <td>1 次</td>
    </tr>
    <tr>
      <td>MMAD + LeakyRelu 串行算子</td>
      <td>约 <code>2.490620 ms</code></td>
      <td><code>KERNEL_AIVEC</code> 约 <code>0.003848 ms</code>，<code>KERNEL_AICORE</code> 约 <code>0.009997 ms</code></td>
      <td>2 次</td>
    </tr>
  </tbody>
</table>

性能对比如下：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <thead>
    <tr>
      <th align="left">对比项</th>
      <th align="left">性能提升</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Launch 端到端耗时</td>
      <td>约 <code>30.00%</code></td>
    </tr>
    <tr>
      <td>Kernel 计算耗时</td>
      <td>约 <code>20.51%</code></td>
    </tr>
  </tbody>
</table>

从纯 kernel 计算时间看，融合实现中的 `KERNEL_MIX_AIC` 与串行实现中 `KERNEL_AICORE + KERNEL_AIVEC` 的计算量级接近，差距并不主要来自矩阵计算或 Vector 计算本身。真正拉开端到端耗时的是 Host 侧算子唤起、参数准备、调度提交、同步等待等固定开销。串行实现需要先 launch MMAD，再 launch LeakyRelu，中间还要经过一次额外的算子调度边界；融合实现只 launch 一次，在同一个 Mix kernel 内完成 Cube 计算、跨核同步和 Vector 后处理，因此减少了第二次算子唤起带来的不必要头开销。且该对比场景中，串行执行样例没有执行多余的D2H和H2D操作，直接复用Mmad算子输出的GM数据作为leakyrelu的输入，若在GM不可复用的场景，性能差距会更大。

算子内部流水如下图所示：

![CV 融合算子内部流水](images/04_02_static_tensor_cv_fusion/5_5_op_pipeline_graph.jpg)

在融合 kernel 内部，AIC 侧依次完成 `MTE2 → MTE1 → Cube → Fixpipe`，将 `MMAD + Bias` 的结果写回 GM；随后 AIV 侧通过跨核同步等待该结果可见，再执行 `MTE2 → Vector → MTE3` 完成 `LeakyRelu` 和写回。也就是说，两个计算阶段虽然分别运行在 Cube 和 Vector 硬件上，但它们处在同一次 kernel launch 的内部流水中，通过核内同步连续衔接，而不是拆成两个独立算子分别由 Host 发起。

CV 融合算子的收益可以概括为：

- 减少 Host 侧 launch 次数，从串行实现的 2 次降为融合实现的 1 次；
- 减少两算子之间的调度、参数处理和同步等待开销；
- 在 kernel 内部用跨核同步连接 AIC 输出和 AIV 输入，使 `MMAD + LeakyRelu` 连续执行；
- 对小算子或 kernel 本身耗时较短的场景，固定唤起开销占比更高，融合带来的端到端收益更明显。



---
# 课后实践

请基于本节 `MMAD + Bias + LeakyRelu` 融合样例，完成 `MMAD + Bias + ResidualAdd + LeakyRelu`。

目标：在 AIV 侧新增一路 Residual 输入，在 LeakyRelu 前完成逐元素 Add，并将融合算子命名为 `mmad_radd_leakyrelu_custom`。

用户需修改如下3个文件：
1. 修改 `mmad_leakyrelu_operator.h`


In [ ]:
%%writefile src/04_02_static_tensor_cv_fusion/mmad_leakyrelu_operator.h


2. 修改 `mmad_leakyrelu_basic_api.asc`


In [ ]:
%%writefile src/04_02_static_tensor_cv_fusion/mmad_leakyrelu_basic_api.asc


3. 修改 `scripts/gen_data.py`


In [ ]:
%%writefile src/04_02_static_tensor_cv_fusion/scripts/gen_data.py


## 实践验证

完成修改后，执行 `run_mmad_leakyrelu.sh` 验证结果。


In [ ]:
!cd src/04_02_static_tensor_cv_fusion && bash run_mmad_leakyrelu.sh --mode=npu --npu-arch=dav-2201


## 参考答案

参考答案如下。


In [ ]:
!cat answer/04_02_static_tensor_cv_fusion/mmad_leakyrelu_operator.h


In [ ]:
!cat answer/04_02_static_tensor_cv_fusion/mmad_leakyrelu_basic_api.asc


In [ ]:
!cat answer/04_02_static_tensor_cv_fusion/scripts/gen_data.py
